In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [2]:
TARGET_WORDS = [
    "überspannen",
    "Manschette",
    "Fuß",
    "Rezeption",
    "abgebrüht",
    "Dynamik",
    "Engpaß",
    "abbauen",
    "Mißklang",
    "Abgesang",
    "Knotenpunkt",
    "Spielball",
    "zersetzen",
    "Armenhaus",
    "Ohrwurm",
    "Eintagsfliege",
    "Seminar",
    "Sensation",
    "Titel",
    "Schmiere",
    "ausspannen",
    "packen",
    "artikulieren",
    "abdecken",
]

In [ ]:
annotations_schemas = {
    "en-en": "schemas_for_german/en-en/german",
    "ru-ru": "schemas_for_german/ru-ru/german",
    "rusemshift-finetune": "schemas_for_german/rusemshift/finetune/german",
    "rusemshift-train": "schemas_for_german/rusemshift/train/german",
}

In [ ]:
def load_scores_for_word(schema_base_path: str, word: str) -> pd.DataFrame:
    score_file = Path(schema_base_path) / word / f"dev.{word}.scores"
    if not score_file.exists():
        print(f"  File not found: {score_file}")
        return None
    
    data = pd.read_json(score_file)
    data["score"] = data["score"].apply(lambda x: np.mean([float(v) for v in x]))
    return data

In [ ]:
def compute_apd(scores_df: pd.DataFrame) -> float:
    if scores_df is None or len(scores_df) == 0:
        return None
    
    return float(scores_df["score"].mean())

In [ ]:
results = {}

for schema_name, schema_path in annotations_schemas.items():
    print(f"\n=== {schema_name} ===")
    results[schema_name] = {}
    for word in TARGET_WORDS:
        scores_df = load_scores_for_word(schema_path, word)
        apd = compute_apd(scores_df)
        results[schema_name][word] = apd
        if apd is not None:
            print(f"  {word}: APD = {apd:.4f}")
        else:
            print(f"  {word}: no data")

In [ ]:
def load_gold_data(path: str) -> dict:
    df = pd.read_csv(path, sep="\t")
    try:
        return dict(zip(df["lemma"], df["change_graded"]))
    except KeyError:
        return dict(zip(df["word"], df["change_graded"]))
    
gold_data = load_gold_data("gold-data-de.csv")

print("Spearman correlations: ")
for schema_name in annotations_schemas:
    pairs = [
        (results[schema_name][w], gold_data[w])
        for w in TARGET_WORDS
        if results[schema_name].get(w) is not None and w in gold_data
    ]
    if len(pairs) < 2:
        print(f"  {schema_name}: not enough data")
        continue
    
    apd_values, gold_values = zip(*pairs)
    spearman, pvalue = spearmanr(gold_values, apd_values)
    print(f"  {schema_name}: Spearman = {spearman:.4f} (p = {pvalue:.4f})")


In [ ]:
rows = []
for schema_name in annotations_schemas:
    for word in TARGET_WORDS:
        rows.append(
            {
                "schema": schema_name,
                "word": word,
                "apd": results[schema_name].get(word),
            }
        )

summary_df = pd.DataFrame(rows)
summary_df.pivot(nindex="word", columns="schema", values="apd").round(4)